# Analysis-period DA scorecard: compute metrics

Scores a fixed set of variables against references at every verification time of the assimilation window, then derives the tables behind the scorecard heat maps (`05_plot_da_scorecards.ipynb`):

* **Figure 1, S0 with vs without DA:** % change in RMSE of DART10/20/40 relative to CTRL10.
* **Figure 2, DART40 S0 vs S1:** RMSE of DART40-S0 and DART40-S1 normalised by the ERA5/observed variability in each window (std about the grid-point window mean), so errors can be compared across seasons. No CTRL ran during the S1 window, so a with/without-DA change is not available for S1.
* **Update impact (supplementary table):** % change in RMSE from the 6-h forecast (prior) to the analysis, for variables DART updates.

| Group | Rows | Model state | Model source | Reference |
|---|---|---|---|---|
| A Upper-air | T, U, V, Q at 850/500/200 hPa | analysis (and prior) | DART `output_*` / `forecast_*` (remapped) | ERA5 6-hourly |
| B Surface state | PSL, TS, TREFHT, U10 | analysis (PSL also prior) | `ts/6hourly_da` (prior: `ts/6hourly`) | ERA5 6-hourly |
| C Water cycle & fluxes | PRECT (GPCP, IMERG, ERA5), LHFLX, SHFLX | daily mean of 6-h forecast means | `ts/6hourly` | daily GPCP/IMERG; ERA5 mean of 00/06/12/18Z |
| D Radiation | FLUT, FLDS, FSDS | daily mean of 6-h forecast means | `ts/6hourly` | NOAA daily OLR; ERA5 mean of 00/06/12/18Z |

Scores per time and region (cos-latitude weighted, one mask shared by all experiments/states): bias, MSE, ensemble variance, Gaussian CRPS. Window RMSE = sqrt(mean MSE). % changes carry a paired moving-block bootstrap 95% interval (1-day blocks).

**Caveat:** ERA5 flux/precipitation fields are hourly means sampled at 00/06/12/18Z, so their "daily" reference undersamples the diurnal cycle (mostly FSDS/FLDS). The error is common to all experiments, so it largely cancels in DART-vs-CTRL changes but inflates absolute RMSE.

**Verification window (both seasons, every row, every experiment): 64 assimilation cycles.**

| | S0 | S1 |
|---|---|---|
| Analyses | 16 Dec 2011 00Z – 31 Dec 18Z | 16 May 2012 00Z – 31 May 18Z |
| 6-h forecasts (prior) valid | 16 Dec 00Z – 1 Jan 00Z | 16 May 00Z – 1 Jun 00Z |
| Daily rows | 16–31 Dec | 16–31 May |

These are the same cycles as the state-space notebooks (06/07) and the common-observation `obs_diag` products (whose 1 Jan / 1 Jun 00Z bin holds no common observations). The all-observation products in notebooks 03/04 also include the 1 Jan 00Z cycle.

## 1. Settings

In [1]:
from pathlib import Path

INPUT_DATA_ROOT = Path("/compyfs/zhan391/v3_dart_cda_scratch")
WORK_DIR = Path("/compyfs/zhan391/work_package/e3sm_dart_analysis")
DIAGNOSTIC_OUTPUT_ROOT = Path("/compyfs/www/zhan391/e3sm_dart/diag_dart_2026")
workflow_name = "analysis_atm_model_evaluation"
workflow_subdir = "da_scorecard"
# Pressure-level DART summaries (prior/posterior mean and sd), 1 degree.
REMAP_ROOT = DIAGNOSTIC_OUTPUT_ROOT / "data/analysis_atm_init/remapped_ensemble_diagnostics"

# Experiments scored per season (S1 has no CTRL during the analysis period).
seasons = {
    "S0": ("CTRL10-S0", "DART10-S0", "DART20-S0", "DART40-S0"),
    "S1": ("DART40-S1",),
}
# Figure 1: (season, test, baseline); Figure 2: DART40 analysis vs prior per season.
fig1_baseline = ("S0", "CTRL10-S0")
fig2_experiments = {"S0": "DART40-S0", "S1": "DART40-S1"}
n_bootstrap = 1000
force_compute = False

## 2. Project setup

In [2]:
import sys
repo = WORK_DIR.resolve()
if str(repo / "diagnostic") not in sys.path:
    sys.path.insert(0, str(repo / "diagnostic"))
from configs.output_paths import workflow_output_dirs
from util.da_scorecard import (default_rows, compute_scorecard, window_scores, change_table, primary_state,
                               compute_reference_std, normalized_rmse_table)

data_dir, figure_dir = workflow_output_dirs(workflow_name, workflow_subdir, output_root=DIAGNOSTIC_OUTPUT_ROOT)

rows = default_rows()
cache_dir = data_dir / "per_time_scores"
print(f"{len(rows)} rows:", ", ".join(r.name for r in rows))

/qfs/people/zhan391/.conda/envs/e3sm_analysis/lib/python3.12/site-packages/pyproj/network.py:59: UserWarning: pyproj unable to set PROJ database path.
  _set_context_ca_bundle_path(ca_bundle_path)


24 rows: T850, T500, T200, U850, U500, U200, V850, V500, V200, Q850, Q500, Q200, PSL, TS, TREFHT, U10, PRECT (GPCP), PRECT (IMERG), PRECT (ERA5), LHFLX, SHFLX, FLUT (NOAA), FLDS, FSDS


## 3. Per-time scores (cached per season and row)

In [3]:
results, failures = compute_scorecard(
    rows, seasons, data_root=INPUT_DATA_ROOT, remap_root=REMAP_ROOT,
    reference_root=INPUT_DATA_ROOT / "reference", cache_dir=cache_dir,
    force_compute=force_compute,
    log=lambda message: None if message.startswith("  ") else print(message),
)
if failures:
    print(f"\n{len(failures)} row(s) failed:")
    for season, row, error in failures:
        print(f"  {season} {row}: {error}")

[REUSE] scorecard_S0_T850.nc
[REUSE] scorecard_S0_T500.nc
[REUSE] scorecard_S0_T200.nc
[REUSE] scorecard_S0_U850.nc


[REUSE] scorecard_S0_U500.nc
[REUSE] scorecard_S0_U200.nc
[REUSE] scorecard_S0_V850.nc


[REUSE] scorecard_S0_V500.nc
[REUSE] scorecard_S0_V200.nc
[REUSE] scorecard_S0_Q850.nc
[REUSE] scorecard_S0_Q500.nc


[REUSE] scorecard_S0_Q200.nc
[REUSE] scorecard_S0_PSL.nc
[REUSE] scorecard_S0_TS.nc
[REUSE] scorecard_S0_TREFHT.nc


[REUSE] scorecard_S0_U10.nc
[REUSE] scorecard_S0_PRECT_GPCP.nc
[REUSE] scorecard_S0_PRECT_IMERG.nc
[REUSE] scorecard_S0_PRECT_ERA5.nc


...

[REUSE] scorecard_S1_Q850.nc
[REUSE] scorecard_S1_Q500.nc
[REUSE] scorecard_S1_Q200.nc


[REUSE] scorecard_S1_PSL.nc
[REUSE] scorecard_S1_TS.nc
[REUSE] scorecard_S1_TREFHT.nc


[REUSE] scorecard_S1_U10.nc
[REUSE] scorecard_S1_PRECT_GPCP.nc
[REUSE] scorecard_S1_PRECT_IMERG.nc
[REUSE] scorecard_S1_PRECT_ERA5.nc


[REUSE] scorecard_S1_LHFLX.nc
[REUSE] scorecard_S1_SHFLX.nc
[REUSE] scorecard_S1_FLUT_NOAA.nc


[REUSE] scorecard_S1_FLDS.nc
[REUSE] scorecard_S1_FSDS.nc


## 4. Window scores

In [4]:
import pandas as pd
frames = []
for (season, row), ds in results.items():
    frame = window_scores(ds).to_dataframe().reset_index()
    frame = frame.dropna(subset=["rmse"])
    frame.insert(0, "row", row); frame.insert(0, "group", ds.attrs["group"]); frame.insert(0, "season", season)
    frame["units"] = ds.attrs["units"]; frame["verification"] = ds.attrs["verification"]
    frames.append(frame)
window_table = pd.concat(frames, ignore_index=True)
window_table.to_csv(data_dir / "window_scores.csv", index=False)
print(f"[TABLE] {data_dir / 'window_scores.csv'}")
window_table[window_table.region == "Global"].round(3)

[TABLE] /compyfs/www/zhan391/e3sm_dart/diag_dart_2026/data/analysis_atm_model_evaluation/da_scorecard/window_scores.csv


,season,group,row,region,state,experiment,bias,rmse,spread,ssr,crps,units,verification
0,S0,A Upper-air state,T850,Global,analysis,CTRL10-S0,-0.806,3.785,3.199,0.886,1.972,K,6-hourly
1,S0,A Upper-air state,T850,Global,analysis,DART10-S0,-0.488,1.803,0.773,0.450,1.053,K,6-hourly
2,S0,A Upper-air state,T850,Global,analysis,DART20-S0,-0.450,1.659,0.795,0.491,0.939,K,6-hourly
3,S0,A Upper-air state,T850,Global,analysis,DART40-S0,-0.430,1.585,0.810,0.517,0.886,K,6-hourly
4,S0,A Upper-air state,T850,Global,prior,CTRL10-S0,-0.806,3.785,3.199,0.886,1.972,K,6-hourly
...,...,...,...,...,...,...,...,...,...,...,...,...,...
720,S1,C Water cycle & fluxes,LHFLX,Global,forecast,DART40-S1,5.779,38.148,9.569,0.254,22.420,W m-2,daily
724,S1,C Water cycle & fluxes,SHFLX,Global,forecast,DART40-S1,5.514,20.641,4.681,0.230,11.905,W m-2,daily
728,S1,D Radiation,FLUT (NOAA),Global,forecast,DART40-S1,7.124,20.672,9.876,0.484,12.810,W m-2,daily
732,S1,D Radiation,FLDS,Global,forecast,DART40-S1,1.559,20.204,6.664,0.334,12.714,W m-2,daily


## 5. Percent changes for the heat maps

In [5]:
season0, baseline = fig1_baseline
fig1_comparisons, fig2_comparisons = set(), set()
for (season, row), ds in results.items():
    state = primary_state(ds)
    if season == season0:
        for test in seasons[season0]:
            if test != baseline:
                fig1_comparisons.add((season0, test, baseline, state, state))
    if "prior" in ds.state.values and not bool(ds["mse"].sel(state="prior").isnull().all()):
        experiment = fig2_experiments[season]
        fig2_comparisons.add((season, experiment, experiment, "analysis", "prior"))

for name, comparisons in [("fig1_dart_vs_ctrl", fig1_comparisons), ("fig2_update_impact", fig2_comparisons)]:
    for score in ("rmse", "crps"):
        table = change_table(results, comparisons=sorted(comparisons), score=score, n_boot=n_bootstrap)
        path = data_dir / f"{name}_{score}_change.csv"
        table.to_csv(path, index=False)
        print(f"[TABLE] {path} ({len(table)} cells)")

[TABLE] /compyfs/www/zhan391/e3sm_dart/diag_dart_2026/data/analysis_atm_model_evaluation/da_scorecard/fig1_dart_vs_ctrl_rmse_change.csv (288 cells)


[TABLE] /compyfs/www/zhan391/e3sm_dart/diag_dart_2026/data/analysis_atm_model_evaluation/da_scorecard/fig1_dart_vs_ctrl_crps_change.csv (288 cells)


[TABLE] /compyfs/www/zhan391/e3sm_dart/diag_dart_2026/data/analysis_atm_model_evaluation/da_scorecard/fig2_update_impact_rmse_change.csv (104 cells)


[TABLE] /compyfs/www/zhan391/e3sm_dart/diag_dart_2026/data/analysis_atm_model_evaluation/da_scorecard/fig2_update_impact_crps_change.csv (104 cells)


## 6. Normalised RMSE for Figure 2 (DART40, S0 vs S1)

RMSE / reference std, where the std is taken about each grid point's window mean (cos-latitude weighted over the region and window). Values below 1 mean the error is smaller than the natural variability of that window.

In [6]:
reference_stds = compute_reference_std(
    rows, seasons, reference_root=INPUT_DATA_ROOT / "reference",
    cache_path=data_dir / "reference_std.csv", force_compute=force_compute,
)
fig2_table = normalized_rmse_table(results, reference_stds, fig2_experiments)
path = data_dir / "fig2_normalized_rmse.csv"
fig2_table.to_csv(path, index=False)
print(f"[TABLE] {path} ({len(fig2_table)} cells)")
fig2_table[fig2_table.region == "Global"].pivot(index="row", columns="season", values="nrmse").round(2)

[WRITE] reference_std.csv
[TABLE] /compyfs/www/zhan391/e3sm_dart/diag_dart_2026/data/analysis_atm_model_evaluation/da_scorecard/fig2_normalized_rmse.csv (192 cells)


season,S0,S1
row,,
FLDS,0.93,1.18
FLUT (NOAA),0.99,0.98
FSDS,1.01,1.34
LHFLX,0.69,0.98
PRECT (ERA5),0.79,1.07
PRECT (GPCP),0.87,0.88
PRECT (IMERG),0.90,0.93
PSL,1.09,0.93
Q200,0.88,0.89
